# EDO do oscilador forçado

$$m\,x''(t) + c\,x'(t) + k\,x(t) = F_0\sin(\omega t)$$

In [ ]:
from sympy import symbols, Function, Eq, sin, diff, dsolve, init_printing

init_printing()

t = symbols('t')
m, c, k, F0, omega = symbols('m c k F0 omega', positive=True)
x = Function('x')

edo = Eq(
    m*diff(x(t), t, 2) + c*diff(x(t), t) + k*x(t),
    F0*sin(omega*t),
)
edo

In [ ]:
sol = dsolve(edo, x(t))
sol

In [ ]:
from sympy import simplify

simplify(sol.rhs)

## Gráfico de x(t)

Atribuímos valores numéricos aos parâmetros e condições iniciais $x(0)=0$, $x'(0)=0$.

> **Fonte dos valores — GRANDO, D.** *Modelagem de vagão ferroviário em sistema multicorpos para análise dinâmica em via com desnivelamento transversal periódico.* Dissertação (Mestrado) — Escola de Engenharia de São Carlos, USP, São Carlos, 2013.
>
> - $m = 32\,500$ kg — 1/4 da massa da caixa carregada (≈ 129 809 kg, **Tab. 12, p.183**).
> - $k = 1\,908\,100$ N/m — soma das molas helicoidais externa (1 197 600 N/m) e interna (710 500 N/m) do truque Y25 (**Tab. 6, p.105**).
> - $\zeta = 0{,}10$ — amortecimento viscoso equivalente (didático). Valor experimental medido por Grando para o modo vertical: $\zeta \approx 0{,}28\%$ (p.132), dominado por atrito seco.
> - $F_0 = 22\,800$ N — equivalente a $k \times 12$ mm de irregularidade vertical típica da via.
> - $\omega = 6$ rad/s — próximo da frequência natural $\omega_n \approx 7{,}66$ rad/s para evidenciar a amplificação.

PDF da referência disponível em `Referencias/Grando_2013_Modelagem_Vagao_Ferroviario_USP.pdf`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import lambdify

# Valores extraídos / derivados de Grando (2013) - vagão de carga com truque Y25.
# Modelo SDOF: 1/4 do vagão sobre uma suspensão.
zeta_v = 0.10
m_v  = 32_500.0
k_v  = 1_908_100.0
c_v  = 2 * zeta_v * np.sqrt(k_v * m_v)

vals = {m: m_v, c: c_v, k: k_v, F0: 22_800.0, omega: 6.0}

sol_ic = dsolve(
    edo, x(t),
    ics={x(0): 0, x(t).diff(t).subs(t, 0): 0},
)
x_func = lambdify(t, sol_ic.rhs.subs(vals), 'numpy')

ts = np.linspace(0, 15, 2000)
x_vals = np.real_if_close(x_func(ts), tol=1000).real * 1000  # m -> mm

plt.figure(figsize=(9, 4))
plt.plot(ts, x_vals)
plt.xlabel('t  [s]')
plt.ylabel('x(t)  [mm]')
plt.title(r'Vibração vertical (1/4 do vagão)  $m\ddot{x}+c\dot{x}+kx=F_0\sin(\omega t)$')
plt.grid(True)
plt.show()

print(f"omega_n = {np.sqrt(k_v/m_v):.3f} rad/s  ({np.sqrt(k_v/m_v)/(2*np.pi):.3f} Hz)")
print(f"c       = {c_v:.0f} N.s/m  (zeta = {zeta_v})")

## Ângulo de fase $\varphi(\omega)$ — o **atraso da resposta**

### O que é

O ângulo de fase $\varphi$ mede o **atraso angular entre a força que sacode o vagão e o deslocamento que ele realmente faz**. Quando a via aplica $F(t) = F_0\sin(\omega t)$, o vagão não responde instantaneamente: leva um pequeno tempo para reagir. Esse atraso, expresso em ângulo, é $\varphi$. A resposta em regime permanente fica

$$x_p(t) = A\,\sin(\omega t - \varphi)$$

— o sinal de menos é o "atraso" do deslocamento em relação à força.

### Analogia do balanço

Imagine empurrar uma criança num balanço:

| Como você empurra | $\varphi$ que aparece | O que acontece |
|---|---|---|
| Muito devagar  ($\omega \ll \omega_n$) | $\approx 0°$ | O balanço segue sua mão; empurrou para frente, ele vai para frente. |
| No ritmo natural ($\omega = \omega_n$) | $90°$ | Você empurra exatamente quando o balanço passa pelo ponto mais baixo. É a **ressonância** — energia entra com eficiência máxima. |
| Muito rápido ($\omega \gg \omega_n$) | $\to 180°$ | O balanço fica em oposição à sua mão; você empurra para frente quando ele já vem para trás. |

### Como se calcula

Da seção 6.2 do trabalho:

$$\tan\varphi = \frac{c\,\omega}{k - m\,\omega^2}$$

O **amortecimento $c$** no numerador é o que cria a **transição suave** entre os três regimes. Sem amortecimento ($c=0$), $\varphi$ saltaria direto de $0°$ para $180°$ ao passar por $\omega_n$, sem o "meio do caminho" em $90°$.

Como o denominador $(k - m\omega^2)$ troca de sinal em $\omega = \omega_n = \sqrt{k/m}$, o código usa `np.arctan2` (em vez de `arctan` simples) para obter $\varphi \in [0°, 180°]$ corretamente.

### Por que importa no vagão

A curva $\varphi(\omega)$ é uma "assinatura" da suspensão: o ponto onde ela passa por $90°$ marca **a frequência natural $\omega_n$**, ou seja, **a velocidade de tráfego e o comprimento de irregularidade da via que devem ser evitados** — porque é onde a vibração explode (ressonância, seção 10 do trabalho).

> **Parâmetros usados no gráfico abaixo** — mesmos da célula anterior, com origem em **Grando (2013)**: $m = 32\,500$ kg (Tab. 12), $k = 1\,908\,100$ N/m (Tab. 6), $\zeta = 0{,}10$ (didático).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m_v    = 32_500.0       # kg
k_v    = 1_908_100.0    # N/m
zeta   = 0.10
c_v    = 2 * zeta * np.sqrt(k_v * m_v)

omega_n = np.sqrt(k_v / m_v)

omegas  = np.linspace(0.01, 4 * omega_n, 1000)
phi     = np.arctan2(c_v * omegas, k_v - m_v * omegas**2)
phi_deg = np.degrees(phi)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(omegas / omega_n, phi_deg, linewidth=2)
ax.axvline(1.0, color='red', linestyle='--', alpha=0.6,
           label=r'$\omega = \omega_n$ (ressonância)')
ax.axhline(90,  color='gray', linestyle=':', alpha=0.6)
ax.axhline(180, color='gray', linestyle=':', alpha=0.6)
ax.set_xlabel(r'$\omega / \omega_n$  (frequência normalizada)')
ax.set_ylabel(r'$\varphi$  [graus]')
ax.set_title(rf'Ângulo de fase  ($\zeta$ = {zeta:.2f},  '
             rf'$\omega_n$ = {omega_n:.2f} rad/s $\approx$ {omega_n/(2*np.pi):.2f} Hz)')
ax.set_yticks([0, 45, 90, 135, 180])
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Visualização da defasagem no tempo

Aqui plotamos juntos a **força de excitação** $F(t) = F_0\sin(\omega t)$ e a **resposta** $x(t)$ ao longo do tempo. O ângulo de fase $\varphi$ aparece como **deslocamento horizontal** entre os picos da força (linhas tracejadas vermelhas) e os picos do deslocamento (linhas tracejadas azuis).

A defasagem em segundos vale $\Delta t = \varphi / \omega$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Mesmos parâmetros (Grando, 2013)
zeta_v  = 0.10
m_v     = 32_500.0
k_v     = 1_908_100.0
c_v     = 2 * zeta_v * np.sqrt(k_v * m_v)
F0_v    = 22_800.0
omega_v = 6.0

phi = np.arctan2(c_v * omega_v, k_v - m_v * omega_v**2)

ts = np.linspace(0, 15, 2000)
F_vals = F0_v * np.sin(omega_v * ts) / 1000          # kN
x_vals = np.real_if_close(x_func(ts), tol=1000).real * 1000  # mm  (reusa lambdify da célula anterior)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax1.plot(ts, F_vals, color='C3', linewidth=1.5)
ax1.set_ylabel('F(t)  [kN]')
ax1.set_title('Excitação — força periódica da via')
ax1.grid(True, alpha=0.3)

ax2.plot(ts, x_vals, color='C0', linewidth=1.5)
ax2.set_xlabel('t  [s]')
ax2.set_ylabel('x(t)  [mm]')
ax2.set_title(f'Resposta — defasada em $\\varphi$ = {np.degrees(phi):.1f}°  '
              f'($\\Delta t$ = {phi/omega_v*1000:.1f} ms)')
ax2.grid(True, alpha=0.3)

# Marca os picos da F(t) e da resposta x_p(t) em regime permanente
for n in range(15):
    tF = (np.pi/2 + 2*np.pi*n) / omega_v
    tX = (np.pi/2 + phi + 2*np.pi*n) / omega_v
    if tF <= ts[-1]:
        ax1.axvline(tF, color='red',  linestyle='--', alpha=0.4)
    if tX <= ts[-1]:
        ax2.axvline(tX, color='blue', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()